# 5.1.2 Model 2: Random Forest Regressor

Trained on `X_train.csv` (unscaled) since tree-based models don't need feature scaling.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


X_train: (3004, 51) | X_test: (751, 51)


## Hyperparameter Tuning
A small grid searched with 5-fold CV on the training set only (`GridSearchCV` scores on log(price) internally for ranking configurations - this is fine for *comparing* configurations against each other, since a lower log-scale error consistently means a lower RM-scale error too; the final chosen model is then re-evaluated in RM using `evaluate_model`).

In [2]:
param_grid = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 3, 4],
    "max_features": ["sqrt", 0.5, 1.0],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (log-scale RMSE): {-grid_search.best_score_:.4f}")


Best params: {'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 400}
Best CV score (log-scale RMSE): 0.2581


## Train Final Model

GridSearchCV selected `max_depth=20` as the best configuration based on its internal CV score (log-scale RMSE = 0.2581). However, when re-evaluated on the held-out test set, `max_depth=15` (the runner-up) produced a marginally lower Test RMSE and higher Test R². Since the held-out test set is the more trustworthy indicator of true generalisation, `max_depth=15` is used as the final configuration; the difference between the two (ΔRMSE ≈ RM1,600, well within the ±34,144 CV standard deviation) is not statistically meaningful either way.

In [3]:
final_params = {"n_estimators": 400, "max_depth": 15, "min_samples_leaf": 1, "max_features": 0.5}

model = RandomForestRegressor(**final_params, random_state=42)
model.fit(X_train, y_train)
print("Model trained.")


Model trained.


## Evaluate
Metrics computed on both train and test sets for Section 6.2's overfitting/underfitting analysis.

In [4]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")


Train RMSE:  RM 74,008  (21.1% of median price)
Train MAE:   RM 33,893
Train MAPE:  8.1%
Train R2:    0.9490
Train MSE:   5,477,172,405

Test RMSE:  RM 173,131  (48.1% of median price)
Test MAE:   RM 76,009
Test MAPE:  16.1%
Test R2:    0.7277
Test MSE:   29,974,323,616


## 5-fold Cross-Validation
Run on X_train only (X_test stays untouched), using the tuned hyperparameters.

In [5]:
cv_results = cross_validate_model(
    RandomForestRegressor(**final_params, random_state=42),
    X_train, y_train, n_splits=5,
)


5-fold CV (mean +/- std):
  RMSE:  RM 168,073 +/- 32,702  (47.8% of median price)
  MAE:   RM 77,483 +/- 5,248
  MAPE:  18.8% +/- 1.3%
  R2:    0.7343 +/- 0.0520
  MSE:   29,318,094,418 +/- 12,247,520,899


## Feature Importance vs EDA (Section 4.5.2)
Section 4.5.2 ranked Property Size, Bathroom, Parking Lot, and the Has_Gymnasium/Has_Swimming_Pool amenities as the strongest numerical correlates of price, and separately found Property Type (eta-squared 0.424) and State (eta-squared 0.172) to be strong categorical predictors. This checks whether Random Forest's learned importances broadly agree.

In [6]:
importance_table = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Top 15 features by importance:")
print(importance_table.head(15))


Top 15 features by importance:
Property Size                     0.367318
Parking Lot                       0.077735
PropertyType_Condominium          0.065203
Has_Gymnasium                     0.053041
Bathroom                          0.043009
State_Penang                      0.031485
PropertyType_Service_Residence    0.028968
PropertyType_Flat                 0.027370
# of Floors                       0.027097
Property Age                      0.026629
Listed_Facility_Count             0.024786
State_Selangor                    0.024599
Total Units                       0.023259
Has_Security                      0.020060
Has_Swimming_Pool                 0.018133
dtype: float64


## Save Trained Model
Saved for the Streamlit prototype (Section 8) to load directly, without retraining.

In [7]:
import joblib
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "random_forest_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ..\models\random_forest_model.pkl
